In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# 1. Criação da Tabela usando spark.sql com try_cast para tratar dados malformados
# Utilizamos TRY_CAST para que valores de texto em colunas numéricas virem NULL em vez de erro
spark.sql(f"""
CREATE OR REPLACE TABLE leitos_hospitalares
USING DELTA
PARTITIONED BY (COMP)
AS
SELECT
    _c0  AS COMP,                    -- Ano mês de competência
    _c1  AS REGIAO,                  -- Região
    _c2  AS UF,                      -- Estado
    _c3  AS MUNICIPIO,               -- Município
    _c4  AS MOTIVO_DESABILITACAO,    -- Motivo de desabilitação
    _c5  AS CNES,                    -- Código do Cadastro Nacional (CNES)
    _c6  AS NOME_ESTABELECIMENTO,    -- Nome Fantasia
    _c7  AS RAZAO_SOCIAL,            -- Razão Social
    _c8  AS TP_GESTAO,               -- M-Municipal, E-Estadual, D-Dupla, S-Sem Gestão
    _c9  AS CO_TIPO_UNIDADE,         -- Código do tipo de unidade
    _c10 AS DS_TIPO_UNIDADE,         -- Descrição do tipo de unidade
    _c11 AS NATUREZA_JURIDICA,       -- Código da Natureza Jurídica
    _c12 AS DESC_NATUREZA_JURIDICA,  -- Descrição da Natureza Jurídica
    _c13 AS NO_LOGRADOURO,           -- Logradouro
    _c14 AS NU_ENDERECO,             -- Endereço
    _c15 AS NO_COMPLEMENTO,          -- Complemento de endereço
    _c16 AS NO_BAIRRO,               -- Bairro
    _c17 AS CO_CEP,                  -- Código de Endereçamento Postal
    _c18 AS NU_TELEFONE,             -- Telefone
    _c19 AS NO_EMAIL,                -- E-mail
    TRY_CAST(_c20 AS INT) AS LEITOS_EXISTENTE,     -- Quantidade de Leitos Existentes
    TRY_CAST(_c21 AS INT) AS LEITOS_SUS,           -- Quantidade de Leitos SUS
    TRY_CAST(_c22 AS INT) AS UTI_TOTAL_EXIST,      -- Somatório de leitos UTI Existentes
    TRY_CAST(_c23 AS INT) AS UTI_TOTAL_SUS,        -- Somatório de leitos UTI SUS
    TRY_CAST(_c24 AS INT) AS UTI_ADULTO_EXIST,     -- Adulto (I, II e III)
    TRY_CAST(_c25 AS INT) AS UTI_ADULTO_SUS,       -- Adulto (I, II e III)
    TRY_CAST(_c26 AS INT) AS UTI_PEDIATRICO_EXIST, -- Pediátrico (I, II e III)
    TRY_CAST(_c27 AS INT) AS UTI_PEDIATRICO_SUS,   -- Pediátrico (I, II e III)
    TRY_CAST(_c28 AS INT) AS UTI_NEONATAL_EXIST,   -- Neonatal (I, II e III)
    TRY_CAST(_c29 AS INT) AS UTI_NEONATAL_SUS,     -- Neonatal (I, II e III)
    TRY_CAST(_c30 AS INT) AS UTI_QUEIMADO_EXIST,   -- Queimado
    TRY_CAST(_c31 AS INT) AS UTI_QUEIMADO_SUS,     -- Queimado
    TRY_CAST(_c32 AS INT) AS UTI_CORONARIANA_EXIST, -- Coronariana (II e III)
    TRY_CAST(_c33 AS INT) AS UTI_CORONARIANA_SUS    -- Coronariana (II e III)
FROM read_files(
    '/Volumes/workspace/default/raw/fact_hospitais/*.csv',
    format => 'csv',
    delimiter => ';',
    header => 'false',
    encoding => 'ISO-8859-1'
)
""")

# 2. Documentação das colunas baseada nos metadados do PDF (Páginas 1 e 2)
column_comments = {
    "COMP": "Ano mês de competência [cite: 1]",
    "REGIAO": "Região [cite: 1]",
    "UF": "Estado [cite: 1]",
    "MUNICIPIO": "Município [cite: 1]",
    "MOTIVO_DESABILITACAO": "Motivo de desabilitação da unidade [cite: 1]",
    "CNES": "Código do Cadastro Nacional dos Estabelecimentos de Saúde (CNES) [cite: 1]",
    "NOME_ESTABELECIMENTO": "Nome Fantasia [cite: 1]",
    "RAZAO_SOCIAL": "Razão Social [cite: 1]",
    "TP_GESTAO": "M-Municipal, E-Estadual, D-Dupla, S-Sem Gestão [cite: 1]",
    "CO_TIPO_UNIDADE": "Código do tipo de unidade [cite: 1]",
    "DS_TIPO_UNIDADE": "Descrição do tipo de unidade [cite: 1]",
    "NATUREZA_JURIDICA": "Código da Natureza Jurídica do Estabelecimento [cite: 1]",
    "DESC_NATUREZA_JURIDICA": "Descrição da Natureza Jurídica do Estabelecimento [cite: 1]",
    "NO_LOGRADOURO": "Logradouro [cite: 1]",
    "NU_ENDERECO": "Endereço [cite: 1]",
    "NO_COMPLEMENTO": "Complemento de endereço [cite: 1]",
    "NO_BAIRRO": "Bairro [cite: 1]",
    "CO_CEP": "Código de Endereçamento Postal [cite: 1]",
    "NU_TELEFONE": "Telefone [cite: 1]",
    "NO_EMAIL": "E-mail [cite: 1]",
    "LEITOS_EXISTENTE": "Quantidade de Leitos Existentes [cite: 1]",
    "LEITOS_SUS": "Quantidade de Leitos SUS [cite: 2]",
    "UTI_TOTAL_EXIST": "Quantidade de Leitos UTI - Existentes. Somatório de leitos (Adulto, Pediátrico, Neonatal, Queimados e Coronariana) [cite: 2]",
    "UTI_TOTAL_SUS": "Quantidade de Leitos UTI - SUS. Somatório de leitos (Adulto, Pediátrico, Neonatal, Queimados e Coronariana) [cite: 2]",
    "UTI_ADULTO_EXIST": "Quantidade de Leitos Existentes Adulto (I, II e III) [cite: 2]",
    "UTI_ADULTO_SUS": "Quantidade de Leitos SUS Adulto (I, II e III) [cite: 2]",
    "UTI_PEDIATRICO_EXIST": "Quantidade de Leitos Existentes Pediátrico (I, II e III) [cite: 2]",
    "UTI_PEDIATRICO_SUS": "Quantidade de Leitos SUS Pediátrico (I, II e III) [cite: 2]",
    "UTI_NEONATAL_EXIST": "Quantidade de Leitos Existentes Neonatal (I, II e III) [cite: 2]",
    "UTI_NEONATAL_SUS": "Quantidade de Leitos SUS Neonatal (I, II e III) [cite: 2]",
    "UTI_QUEIMADO_EXIST": "Quantidade de Leitos Existentes Queimado [cite: 2]",
    "UTI_QUEIMADO_SUS": "Quantidade de Leitos SUS Queimado [cite: 2]",
    "UTI_CORONARIANA_EXIST": "Quantidade de Leitos Existentes Coronariana (II e III) [cite: 2]",
    "UTI_CORONARIANA_SUS": "Quantidade de Leitos SUS Coronariana (II e III) [cite: 2]"
}

for col_name, comment in column_comments.items():
    spark.sql(f"ALTER TABLE leitos_hospitalares ALTER COLUMN {col_name} COMMENT '{comment}'")

print("Tabela leitos_hospitalares criada com PySpark e documentada com sucesso!")